# Customer Churn Analysis for Telecom Industry
Using IBM Telco dataset

In [ ]:

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# Load data
df = pd.read_csv('../data/telco_churn.csv')
df.head()


## Data Cleaning & Preprocessing

In [ ]:

# Convert TotalCharges to numeric and drop missing values
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df.dropna()

# Encode categorical columns
df.drop('customerID', axis=1, inplace=True)
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
cat_cols = df.select_dtypes(include='object').columns

le = LabelEncoder()
for col in cat_cols:
    df[col] = le.fit_transform(df[col])


## Model Building

In [ ]:

# Feature scaling
X = df.drop('Churn', axis=1)
y = df['Churn']
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Train Random Forest Classifier
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

# Evaluation
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))


## Save Predictions

In [ ]:

# Predict on full dataset
probs = model.predict_proba(X_scaled)[:, 1]
df_out = pd.read_csv('../data/telco_churn.csv')
df_out = df_out.dropna(subset=["TotalCharges"])
df_out['Churn_Probability'] = probs
df_out['Predicted_Churn'] = (probs > 0.5).astype(int)

df_out[['customerID', 'Churn_Probability', 'Predicted_Churn']].to_csv('../outputs/churn_predictions.csv', index=False)


## Confusion Matrix Visualization

In [ ]:

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['No', 'Yes'], yticklabels=['No', 'Yes'])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.savefig('../outputs/visuals/confusion_matrix.png')
plt.show()
